In [1]:
import os
import json
import joblib
import numpy as np
import torch
import torch.nn as nn

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
from groq import Groq

In [7]:
import os
import joblib

PROJECT_DIR = r"C:\Users\Administrator\Downloads\NLP_Final_Project"
MODELS_DIR = os.path.join(PROJECT_DIR, "models")

# Language Detection
language_model = joblib.load(
    os.path.join(MODELS_DIR, "language_detector_model.pkl")
)

language_vectorizer = joblib.load(
    os.path.join(MODELS_DIR, "language_detector_vectorizer.pkl")
)

# Intent Classification
intent_model = joblib.load(
    os.path.join(MODELS_DIR, "intent_model.pkl")
)

intent_vectorizer = joblib.load(
    os.path.join(MODELS_DIR, "intent_vectorizer.pkl")
)

print("All classification models loaded successfully!")

All classification models loaded successfully!


In [8]:
import os

PROJECT_DIR = r"C:\Users\Administrator\Downloads\NLP_Final_Project"

print("All model/artifact files:\n")

for root, dirs, files in os.walk(PROJECT_DIR):
    for file in files:
        if file.endswith((
            ".joblib",
            ".pkl",
            ".pt",
            ".pth",
            ".json",
            ".bin"
        )):
            print(os.path.join(root, file))

All model/artifact files:

C:\Users\Administrator\Downloads\NLP_Final_Project\best_sentiment_model.pt
C:\Users\Administrator\Downloads\NLP_Final_Project\models\intent_model.pkl
C:\Users\Administrator\Downloads\NLP_Final_Project\models\intent_vectorizer.pkl
C:\Users\Administrator\Downloads\NLP_Final_Project\models\language_detector_model.pkl
C:\Users\Administrator\Downloads\NLP_Final_Project\models\language_detector_vectorizer.pkl


In [11]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def detect_language(text):
    cleaned = clean_text(text)
    features = language_vectorizer.transform([cleaned])
    return language_model.predict(features)[0]


def predict_intent(text):
    cleaned = clean_text(text)
    features = intent_vectorizer.transform([cleaned])
    return intent_model.predict(features)[0]


print(detect_language("Where is my order?"))
print(predict_intent("Where is my order?"))

en
ORDER


In [12]:
def detect_language(text):
    cleaned = clean_text(text)
    features = language_vectorizer.transform([cleaned])
    prediction = language_model.predict(features)[0]
    return prediction

In [13]:
def predict_intent(text):
    cleaned = clean_text(text)
    features = intent_vectorizer.transform([cleaned])
    prediction = intent_model.predict(features)[0]
    return prediction

In [14]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Loading RAG dataset...")

rag_dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

rag_df = rag_dataset["train"].to_pandas()

documents = [
    f"Customer issue: {row['instruction']}\n"
    f"Support response: {row['response']}\n"
    f"Intent: {row['category']}"
    for _, row in rag_df.iterrows()
]

print("Documents:", len(documents))

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("FAISS index ready!")
print("Vectors:", index.ntotal)

Loading RAG dataset...
Documents: 26872


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/840 [00:00<?, ?it/s]

FAISS index ready!
Vectors: 26872


In [15]:
def retrieve_documents(query, k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "document": documents[idx],
            "score": float(score)
        })

    return results

In [16]:
results = retrieve_documents("Where is my order?", k=3)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Score:", result["score"])
    print(result["document"][:500])


--- Result 1 ---
Score: 0.6795310974121094
Customer issue: where do I order something?
Support response: I'm delighted to assist you in finding the right place to order our products! To place an order, you can visit our website at {{Website URL}}. Our website is designed to provide you with a user-friendly and seamless ordering experience. Simply search for the items you wish to purchase, add them to your cart, and proceed to the checkout page to complete your order. Our website also offers secure payment options to ensure a smooth and p

--- Result 2 ---
Score: 0.6486217975616455
Customer issue: where do I order some items?
Support response: Thank you for your question regarding where to order items. I'm here to assist you with that. To place an order, you can visit our online store on our website or use our mobile app. Simply browse through the available products, select the items you want to purchase, and proceed to the checkout page. You'll be prompted to provide your shipping inf

In [17]:
test_messages = [
    "Where is my order?",
    "I want a refund",
    "My payment failed",
    "I forgot my password"
]

for msg in test_messages:
    print("=" * 50)
    print("Message:", msg)
    print("Language:", detect_language(msg))
    print("Intent:", predict_intent(msg))

Message: Where is my order?
Language: en
Intent: ORDER
Message: I want a refund
Language: en
Intent: REFUND
Message: My payment failed
Language: fr
Intent: PAYMENT
Message: I forgot my password
Language: en
Intent: ACCOUNT


In [18]:
def route_message(message):

    language = detect_language(message)
    intent = predict_intent(message)

    if intent in ["greeting", "goodbye", "gratitude"]:
        route = "direct"

    else:
        route = "rag"

    return {
        "message": message,
        "language": language,
        "intent": intent,
        "route": route
    }

In [19]:
messages = [
    "Hello",
    "Where is my order?",
    "I want a refund",
    "My payment was declined",
    "Thank you"
]

for message in messages:
    print(route_message(message))

{'message': 'Hello', 'language': 'it', 'intent': 'ACCOUNT', 'route': 'rag'}
{'message': 'Where is my order?', 'language': 'en', 'intent': 'ORDER', 'route': 'rag'}
{'message': 'I want a refund', 'language': 'en', 'intent': 'REFUND', 'route': 'rag'}
{'message': 'My payment was declined', 'language': 'en', 'intent': 'PAYMENT', 'route': 'rag'}
{'message': 'Thank you', 'language': 'zh', 'intent': 'ACCOUNT', 'route': 'rag'}


In [ ]:
from groq import Groq

api_key = "YOUR_GROQ_API_KEY"

client = Groq(api_key=api_key)

print("Groq connected successfully!")

Groq connected successfully!


In [21]:
SIMILARITY_THRESHOLD = 0.45


def clean_response(text):
    text = text.replace("{{Customer Support Phone Number}}", "")
    text = text.replace("{{Website URL}}", "")
    return text.strip()


def rag_answer(query):

    results = retrieve_documents(query, k=3)

    best_score = results[0]["score"]

    if best_score < SIMILARITY_THRESHOLD:
        return (
            "I don't have enough information in the support "
            "knowledge base to answer this question accurately."
        )

    context = "\n\n---\n\n".join(
        result["document"]
        for result in results
    )

    prompt = f"""
You are an e-commerce customer support assistant.

Answer the customer's question using ONLY the information
provided in the context.

Do not invent policies, prices, phone numbers, URLs,
or any other information.

Context:
{context}

Customer question:
{query}

Give a concise and helpful answer.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful e-commerce customer support assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return clean_response(
        response.choices[0].message.content
    )

In [22]:
def chatbot(message):

    # 1. Language Detection
    language = detect_language(message)

    # 2. Intent Classification
    intent = predict_intent(message)

    # 3. Routing
    if intent in ["greeting"]:
        route = "direct"
        response = "Hello! How can I help you today?"

    elif intent in ["goodbye"]:
        route = "direct"
        response = "Goodbye! Have a great day."

    elif intent in ["gratitude"]:
        route = "direct"
        response = "You're welcome! I'm happy to help."

    else:
        route = "rag"
        response = rag_answer(message)

    return {
        "message": message,
        "language": language,
        "intent": intent,
        "route": route,
        "response": response
    }

In [23]:
test_messages = [
    "Hello",
    "Where is my order?",
    "I want a refund for my purchase.",
    "My payment was declined.",
    "I forgot my password.",
    "Thank you for your help.",
    "Who is the president of Egypt?"
]

for message in test_messages:

    print("\n" + "=" * 80)

    result = chatbot(message)

    print("User:", result["message"])
    print("Language:", result["language"])
    print("Intent:", result["intent"])
    print("Route:", result["route"])
    print("Response:", result["response"])


User: Hello
Language: it
Intent: ACCOUNT
Route: rag
Response: I don't have enough information in the support knowledge base to answer this question accurately.

User: Where is my order?
Language: en
Intent: ORDER
Route: rag
Response: To see the status and location of your order, please log into your account on our website and go to the **Order History** section. There you’ll find details and the current status of all your orders. If you need further help, just let us know!

User: I want a refund for my purchase.
Language: en
Intent: REFUND
Route: rag
Response: To get a refund, first gather your order details (order number, purchase date, and reason for the refund). Then contact our customer‑support team—either by phone at **** or via the Live Chat on our website at ****. They’ll guide you through the refund process and let you know any eligibility requirements.

User: My payment was declined.
Language: en
Intent: PAYMENT
Route: rag
Response: I’m sorry to hear your payment was declined

In [24]:
test_messages = [
    "Hello",
    "Where is my order?",
    "I want a refund",
    "My payment was declined",
    "I forgot my password",
    "Thank you for your help.",
    "Who is the president of Egypt?"
]

for message in test_messages:
    result = chatbot(message)

    print("=" * 80)
    print("User:", result["message"])
    print("Language:", result["language"])
    print("Intent:", result["intent"])
    print("Route:", result["route"])
    print("Bot:", result["response"])

User: Hello
Language: it
Intent: ACCOUNT
Route: rag
Bot: I don't have enough information in the support knowledge base to answer this question accurately.
User: Where is my order?
Language: en
Intent: ORDER
Route: rag
Bot: To see the status of your order, log into your account on our website and go to the **Order History** section. There you’ll find the details and current location of your order. If you need any further help, just let us know!
User: I want a refund
Language: en
Intent: REFUND
Route: rag
Bot: To start a refund, gather your order number, purchase date, and the reason for the refund. Then contact our customer‑support team either by phone at **** or via the Live Chat on our website at ****. They’ll guide you through the next steps and process your request.
User: My payment was declined
Language: en
Intent: PAYMENT
Route: rag
Bot: I’m sorry your payment was declined. To help us investigate, could you please share the date and time of the transaction and any error message yo